In [23]:
import pandas as pd
import numpy as np
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [27]:
df = pd.read_csv("IMDB Dataset.csv")

print("Dataset shape:", df.shape)
print(df.head())

# ==============================
# 2. Text Cleaning Function
# ==============================
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub('<.*?>', '', text)          # remove HTML tags
    text = re.sub('[^a-zA-Z]', ' ', text)     # keep only letters
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

# Apply cleaning
df['cleaned_review'] = df['review'].apply(clean_text)

# ==============================
# 3. Convert Labels
# ==============================
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# ==============================
# 4. TF-IDF Vectorization
# ==============================
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['cleaned_review'])
y = df['sentiment']

# ==============================
# 5. Train-Test Split
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================
# 6. Train Model
# ==============================
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

# ==============================
# 7. Evaluation
# ==============================
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ==============================
# 8. Predict Custom Input
# ==============================
def predict_sentiment(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])
    pred = model.predict(vec)
    return "Positive 😊" if pred[0] == 1 else "Negative 😠"

# Test examples
print("\nCustom Predictions:")
print(predict_sentiment("This movie was absolutely amazing!"))
print(predict_sentiment("Worst movie ever, waste of time."))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dataset shape: (50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Accuracy: 0.8923

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.88      0.91      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Custom Predictions:
Positive 😊
Negative 😠
